In [ ]:
logger.info(\"\\n\" + \"=\"*60)\nlogger.info(\"PIPELINE 00 SUMMARY\")\nlogger.info(\"=\"*60)\nlogger.info(f\"Training samples: {len(train_df)}\")\nlogger.info(f\"Test samples: {len(test_df)}\")\nlogger.info(f\"Classes: {config.num_classes}\")\nlogger.info(f\"Image dimensions: ~{int(stats['train']['avg_height'])}x{int(stats['train']['avg_width'])}px\")\nlogger.info(f\"Data validation: {'PASSED' if validation_results['is_valid'] else 'WARNINGS'}\")\nlogger.info(f\"Missing files: {missing_train} train, {missing_test} test\")\nlogger.info(\"\\nNext step: Run Notebook 01 - Preprocessing & Augmentation\")\nlogger.info(\"=\"*60 + \"\\n\")\n\nprint(\"✓ Pipeline 00 completed successfully!\")"


## 9. Summary

In [ ]:
# Save processed data
logger.info(\"Saving processed data...\")\ntrain_df.to_parquet('processed/train_metadata.parquet')\ntest_df.to_parquet('processed/test_metadata.parquet')\n\n# Save statistics\nwith open('processed/data_stats.json', 'w') as f:\n    json.dump(stats, f, indent=2)\n\n# Save source mapping\nwith open('processed/source_mapping.json', 'w') as f:\n    json.dump(source_mapping, f, indent=2)\n\nlogger.info(\"✓ Saved train_metadata.parquet\")\nlogger.info(\"✓ Saved test_metadata.parquet\")\nlogger.info(\"✓ Saved data_stats.json\")\nlogger.info(\"✓ Saved source_mapping.json\")\n\nprint(\"\\n--- Files Saved ---\")\nprint(f\"  processed/train_metadata.parquet ({train_df.shape[0]} rows, {train_df.shape[1]} cols)\")\nprint(f\"  processed/test_metadata.parquet ({test_df.shape[0]} rows, {test_df.shape[1]} cols)\")\nprint(f\"  processed/data_stats.json\")\nprint(f\"  processed/source_mapping.json\")"


## 8. Save Processed Data and Statistics

In [ ]:
# Analyze formats and color modes\nprint(\"\\n--- Image Formats ---\")\nfor fmt, count in train_df['format'].value_counts().items():\n    print(f\"  {fmt}: {count} ({100*count/len(train_df):.1f}%)\")\n\nprint(\"\\n--- Color Modes ---\")\nfor mode, count in train_df['color_mode'].value_counts().items():\n    print(f\"  {mode}: {count} ({100*count/len(train_df):.1f}%)\")"


In [ ]:
# Analyze image dimensions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Height distribution
axes[0].hist(train_df['height'].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Height (px)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Height Distribution\\n(mean: {train_df[\"height\"].mean():.0f}px)')
axes[0].grid(axis='y', alpha=0.3)

# Width distribution
axes[1].hist(train_df['width'].dropna(), bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Width (px)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Width Distribution\\n(mean: {train_df[\"width\"].mean():.0f}px)')
axes[1].grid(axis='y', alpha=0.3)

# File size distribution (MB)
file_size_mb = train_df['file_size_bytes'] / (1024 * 1024)
axes[2].hist(file_size_mb.dropna(), bins=50, color='green', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('File Size (MB)')
axes[2].set_ylabel('Frequency')
axes[2].set_title(f'File Size Distribution\\n(mean: {file_size_mb.mean():.3f}MB)')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('processed/image_dimensions.png', dpi=100, bbox_inches='tight')
plt.show()

print(\"Image dimension visualization saved\")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Class distribution bar plot
class_counts = train_df['y'].value_counts().sort_index()
axes[0].bar(class_counts.index, class_counts.values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Class ID')
axes[0].set_ylabel('Count')
axes[0].set_title('Training Data: Class Distribution')
axes[0].grid(axis='y', alpha=0.3)

# Class distribution by source name
source_counts = train_df['source_name'].value_counts()
axes[1].barh(range(len(source_counts)), source_counts.values, color='coral', alpha=0.7)
axes[1].set_yticks(range(len(source_counts)))
axes[1].set_yticklabels(source_counts.index, fontsize=9)
axes[1].set_xlabel('Count')
axes[1].set_title('Training Data: Count by Generator')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('processed/class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(\"\\nClass distribution visualization saved\")

In [ ]:
# Compute statistics
stats = data_loader.compute_statistics(train_df, test_df)

print("\\n--- Dataset Statistics ---")
print(f"\\nTraining Set:")
print(f"  Total samples: {stats['train']['total_samples']}")
print(f"  Classes: {stats['train']['classes']}")
print(f"  Average height: {stats['train']['avg_height']:.1f}px")
print(f"  Average width: {stats['train']['avg_width']:.1f}px")
print(f"  Average file size: {stats['train']['avg_file_size_mb']:.2f}MB")

print(f"\\nTest Set:")
print(f"  Total samples: {stats['test']['total_samples']}")
print(f"  Average height: {stats['test']['avg_height']:.1f}px")
print(f"  Average width: {stats['test']['avg_width']:.1f}px")
print(f"  Average file size: {stats['test']['avg_file_size_mb']:.2f}MB")

## 7. Exploratory Data Analysis (EDA)

In [ ]:
# Load source mapping
source_mapping = data_loader.load_source_mapping()

# Add source names to training data
train_df = data_loader.add_source_names(train_df, source_mapping)

print("--- Source Mapping ---")
for class_id, source_name in source_mapping.items():
    print(f"  {class_id}: {source_name}")

print(f"\\n--- Training data with source names ---")
print(train_df[['ID', 'y', 'source_name', 'width', 'height']].head(10))

## 6. Load Source Mapping and Add Names

In [ ]:
# Validate training data
logger.info("Validating training data...")
validation_results = data_loader.validate_training_data(train_df)

print("\\n--- Validation Results ---")
print(f"Valid: {validation_results['is_valid']}")
if validation_results['warnings']:
    print(f"Warnings: {validation_results['warnings']}")
if validation_results['errors']:
    print(f"Errors: {validation_results['errors']}")

print(f"\\nClass distribution: {validation_results['stats']['class_counts']}")

# Check for missing files
missing_train = (train_df['is_readable'] == False).sum()
missing_test = (test_df['is_readable'] == False).sum()

print(f"\\nMissing/corrupted files:")
print(f"  Training: {missing_train}")
print(f"  Test: {missing_test}")

## 5. Validate Data Integrity

In [ ]:
# Resolve paths
logger.info("Resolving file paths...")
train_df = data_loader.resolve_paths(train_df, image_dir='Training')
test_df = data_loader.resolve_paths(test_df, image_dir='Test')

# Extract image metadata
logger.info("Extracting image metadata for training data...")
train_df = data_loader.extract_image_metadata(train_df)

logger.info("Extracting image metadata for test data...")
test_df = data_loader.extract_image_metadata(test_df)

print("\n--- Training Data with Metadata ---")
print(train_df[['ID', 'path', 'y', 'width', 'height', 'format', 'is_readable']].head())
print(f"Shape: {train_df.shape}")

## 4. Resolve Paths and Extract Image Metadata

In [ ]:
# Initialize data loader
data_loader = DataLoader(data_dir=config.data_dir)

# Load metadata
train_df, test_df = data_loader.load_metadata()
print(f"Loaded training data: {len(train_df)} samples")
print(f"Loaded test data: {len(test_df)} samples")

# Display sample
print("\n--- Training Data Sample ---")
print(train_df.head())
print("\n--- Test Data Sample ---")
print(test_df.head())

## 3. Load Training and Test Data

In [ ]:
# Load configuration
config = load_config()

# Create necessary directories
Path(config.data_dir).mkdir(parents=True, exist_ok=True)
Path('processed').mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(parents=True, exist_ok=True)

logger.info(f"Configuration: {config.competition_name}")
logger.info(f"Data directory: {config.data_dir}")
logger.info(f"Number of classes: {config.num_classes}")

## 2. Load Configuration

## 1. Import Libraries and Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Import pipeline modules
from config_loader import load_config
from logger_setup import setup_logger
from data_loader import DataLoader
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Setup logging
logger = setup_logger('pipeline_00', level='INFO')
logger.info("Pipeline 00: Data Loading, Validation & EDA - Started")

# Set random seed for reproducibility
np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Synthetic Image Attribution Challenge - Pipeline 1
## Stage 00: Data Loading, Validation & EDA

This notebook loads the competition data, validates integrity, extracts image metadata, and performs exploratory data analysis.

**Expected Duration:** ~30 minutes  
**Outputs:**
- `processed/train_metadata.parquet` - Training data with metadata
- `processed/test_metadata.parquet` - Test data with metadata
- `processed/data_stats.json` - Dataset statistics
- `logs/pipeline_00.log` - Detailed logs